# Các dòng thiếu `LogonTypeDescription`

Notebook này đọc file `event_4624_day-16.parquet` theo từng batch để giới hạn lượng RAM sử dụng. Một giá trị được xem là thiếu nếu là `null`, chuỗi rỗng, hoặc chỉ chứa khoảng trắng.

In [1]:
from pathlib import Path
from typing import Optional

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

PARQUET_RELATIVE_PATH = Path("data/interim/event_4624/event_4624_day-16.parquet")
TARGET_COLUMN = "LogonTypeDescription"
BATCH_SIZE = 250_000


def find_project_file(relative_path: Path, start_directory: Optional[Path] = None) -> Path:
    """Tìm file từ thư mục hiện tại hoặc một trong các thư mục cha."""
    start_directory = (start_directory or Path.cwd()).resolve()

    for directory in (start_directory, *start_directory.parents):
        candidate = directory / relative_path
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Không tìm thấy '{relative_path}' từ '{start_directory}' hoặc các thư mục cha."
    )


parquet_path = find_project_file(PARQUET_RELATIVE_PATH)
print(f"Đang đọc: {parquet_path}")

Đang đọc: D:\Github Repo\ueba-benchmark\data\interim\event_4624\event_4624_day-16.parquet


In [2]:
def missing_text_mask(values: pa.Array) -> pa.Array:
    """Trả về mask cho null, chuỗi rỗng và chuỗi chỉ có khoảng trắng."""
    values_without_null = pc.fill_null(values, "")
    return pc.equal(pc.utf8_trim_whitespace(values_without_null), "")


def read_rows_with_missing_text(
    file_path: Path,
    column: str,
    batch_size: int = 250_000,
) -> pd.DataFrame:
    """Đọc Parquet theo batch và trả về toàn bộ dòng có giá trị text bị thiếu."""
    parquet_file = pq.ParquetFile(file_path)
    column_names = parquet_file.schema_arrow.names

    if column not in column_names:
        raise KeyError(
            f"Không có cột '{column}'. Các cột hiện có: {column_names}"
        )

    column_index = column_names.index(column)
    matching_frames: list[pd.DataFrame] = []

    for batch in parquet_file.iter_batches(batch_size=batch_size):
        mask = missing_text_mask(batch.column(column_index))
        if pc.any(mask).as_py():
            filtered_batch = pa.Table.from_batches([batch]).filter(mask)
            matching_frames.append(filtered_batch.to_pandas())

    if not matching_frames:
        return pd.DataFrame(columns=column_names)

    return pd.concat(matching_frames, ignore_index=True)


missing_rows = read_rows_with_missing_text(
    parquet_path,
    TARGET_COLUMN,
    batch_size=BATCH_SIZE,
)

In [3]:
print(f"Số dòng thiếu {TARGET_COLUMN}: {len(missing_rows):,}")

if missing_rows.empty:
    print("Không có dòng nào bị thiếu.")
else:
    # to_string() giúp in đủ tất cả dòng và tất cả cột thay vì bị rút gọn.
    print(missing_rows.to_string(index=False))

Số dòng thiếu LogonTypeDescription: 3
   Time  EventID    LogHost  LogonType LogonTypeDescription UserName   DomainName LogonID SubjectUserName SubjectDomainName SubjectLogonID Status Source ServiceName Destination AuthenticationPackage FailureReason  ProcessName ProcessID ParentProcessName ParentProcessID
1323712     4624 Comp590388          5                  NaN   system nt authority   0x3e7             NaN               NaN            NaN    NaN    NaN         NaN         NaN             Negotiate           NaN services.exe     0x32c               NaN             NaN
1323788     4624 Comp590388          5                  NaN   system nt authority   0x3e7             NaN               NaN            NaN    NaN    NaN         NaN         NaN             Negotiate           NaN services.exe     0x32c               NaN             NaN
1329491     4624 Comp590388          5                  NaN   system nt authority   0x3e7             NaN               NaN            NaN    NaN    NaN

In [4]:
NUMBER_OF_EXAMPLES = 10

parquet_file = pq.ParquetFile(parquet_path)
column_names = parquet_file.schema_arrow.names
logon_type_index = column_names.index("LogonType")

example_frames = []
remaining_examples = NUMBER_OF_EXAMPLES

for batch in parquet_file.iter_batches(batch_size=BATCH_SIZE):
    mask = pc.equal(batch.column(logon_type_index), 5)

    if pc.any(mask).as_py():
        matched_table = pa.Table.from_batches([batch]).filter(mask)
        selected_table = matched_table.slice(0, remaining_examples)

        example_frames.append(selected_table.to_pandas())
        remaining_examples -= selected_table.num_rows

    if remaining_examples == 0:
        break

if example_frames:
    logon_type_5_examples = pd.concat(example_frames, ignore_index=True)
    print(f"Một số ví dụ có LogonType = 5 ({len(logon_type_5_examples)} dòng):")
    print(logon_type_5_examples.to_string(index=False))
else:
    print("Không tìm thấy dòng nào có LogonType = 5.")

Một số ví dụ có LogonType = 5 (10 dòng):
   Time  EventID    LogHost  LogonType LogonTypeDescription UserName   DomainName LogonID SubjectUserName SubjectDomainName SubjectLogonID Status Source ServiceName Destination AuthenticationPackage FailureReason  ProcessName ProcessID ParentProcessName ParentProcessID
1296000     4624 Comp995416          5              Service   system nt authority   0x3e7             NaN               NaN            NaN    NaN    NaN         NaN         NaN             Negotiate           NaN services.exe     0x2bc               NaN             NaN
1296000     4624 Comp861045          5              Service   system nt authority   0x3e7             NaN               NaN            NaN    NaN    NaN         NaN         NaN             Negotiate           NaN services.exe     0x2bc               NaN             NaN
1296000     4624 Comp635310          5              Service   system nt authority   0x3e7             NaN               NaN            NaN    NaN    